# OmniCare Clinical Copilot - Notebook 4: Discharge Summary

**Pipeline:** Aggregate All Stages → MedGemma → Comprehensive Discharge Summary

This notebook brings together the entire patient journey:
1. Consultation SOAP note (Notebook 1)
2. Admission note + vitals (Notebook 2)
3. Radiology reports (Notebook 3)

**Prerequisites:** All previous notebooks must have been run for the same encounter_id.

## 1. Setup & Load Encounter

In [ ]:
import os
import sys
import json
import torch
from datetime import datetime

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from utils.encounter_state import load_encounter, update_stage, list_encounters, save_encounter
from utils.fhir_helpers import format_vitals_summary
from utils.mcp_tools import generate_discharge_summary

# Load encounter
available = list_encounters()
print(f"Available encounters: {available}")

encounter_id = available[-1] if available else None
encounter = load_encounter(encounter_id)
print(f"\nLoaded encounter: {encounter_id}")
print(f"Patient: {encounter['patient']['name']}")

# Show which stages are complete
for stage, data in encounter["stages"].items():
    has_data = data.get("timestamp") is not None
    status = "COMPLETE" if has_data else "MISSING"
    print(f"  {stage}: {status}")

## 2. Aggregate All Clinical Documentation

In [ ]:
# Extract data from each stage
consultation = encounter["stages"]["consultation"]
admission = encounter["stages"]["admission"]
radiology = encounter["stages"]["radiology"]

# Format SOAP note
soap = consultation.get("soap_note", {})
soap_text = ""
if isinstance(soap, dict):
    for section in ["subjective", "objective", "assessment", "plan"]:
        content = soap.get(section, "")
        if content:
            soap_text += f"**{section.title()}:** {content}\n\n"
else:
    soap_text = str(soap)

if not soap_text.strip():
    soap_text = "No consultation note available."

# Format admission note
admission_note = admission.get("admission_note", "No admission note available.")

# Format vitals trend
vitals_history = admission.get("vitals_history", [])
vitals_trend = format_vitals_summary(vitals_history)

# Format radiology reports
radiology_reports = ""
for i, report in enumerate(radiology.get("reports", []), 1):
    findings = report.get("findings", "No findings")
    radiology_reports += f"\nReport {i}:\n{findings}\n"

if not radiology_reports.strip():
    radiology_reports = "No radiology reports available."

print("=" * 60)
print("AGGREGATED CLINICAL DATA")
print("=" * 60)
print(f"\n--- SOAP Note ({len(soap_text)} chars) ---")
print(soap_text[:300] + "..." if len(soap_text) > 300 else soap_text)
print(f"\n--- Admission Note ({len(str(admission_note))} chars) ---")
print(str(admission_note)[:300] + "..." if len(str(admission_note)) > 300 else admission_note)
print(f"\n--- Vitals ({len(vitals_history)} readings) ---")
print(vitals_trend[:300])
print(f"\n--- Radiology ({len(radiology.get('reports', []))} reports) ---")
print(radiology_reports[:300])

## 3. Generate Discharge Summary (MedGemma)

In [ ]:
print("Generating comprehensive discharge summary with MedGemma...")
print("This combines all stages: consultation → admission → radiology → discharge.\n")

discharge_summary = generate_discharge_summary(
    soap_note=soap_text,
    admission_note=str(admission_note) if admission_note else "No admission note.",
    vitals_trend=vitals_trend,
    radiology_reports=radiology_reports,
    model=medgemma_model,
    processor=medgemma_processor,
    max_new_tokens=1536
)

print("=" * 60)
print("DISCHARGE SUMMARY")
print("=" * 60)
print(discharge_summary)

## 4. ICD-10 Code Suggestions

Extract diagnoses from the discharge summary and suggest ICD-10 codes.

This uses the ICD-10 MCP tools available in the environment.

In [ ]:
# Extract key diagnoses from the encounter data
# In a full MCP setup, this would call the ICD-10 search_codes tool
conditions = admission.get("conditions", [])
diagnoses = [c.get("display", "") for c in conditions if c.get("display")]

# Also try to extract from SOAP assessment
assessment = soap.get("assessment", "") if isinstance(soap, dict) else ""

print("Diagnoses identified:")
for d in diagnoses:
    print(f"  - {d}")

if assessment:
    print(f"\nAssessment text (for ICD-10 coding):")
    print(f"  {assessment[:300]}")

print("\n--- ICD-10 Code Suggestions ---")
print("(In production, these would be looked up via the ICD-10 MCP tool)")

# Common ICD-10 mappings for the demo scenario
icd10_suggestions = []
diagnosis_map = {
    "pneumonia": {"code": "J18.9", "description": "Pneumonia, unspecified organism"},
    "type 2 diabetes": {"code": "E11.9", "description": "Type 2 diabetes mellitus without complications"},
    "hypertension": {"code": "I10", "description": "Essential (primary) hypertension"},
    "cough": {"code": "R05.9", "description": "Cough, unspecified"},
    "fever": {"code": "R50.9", "description": "Fever, unspecified"},
}

combined_text = (" ".join(diagnoses) + " " + assessment + " " + discharge_summary).lower()
for keyword, code_info in diagnosis_map.items():
    if keyword in combined_text:
        icd10_suggestions.append(code_info)
        print(f"  {code_info['code']}: {code_info['description']}")

if not icd10_suggestions:
    print("  No ICD-10 codes auto-matched. Manual review needed.")

## 5. Save Discharge Summary to Encounter State

In [ ]:
# Extract medication list from admission data
medications = admission.get("medications", [])
meds_at_discharge = [m.get("display", "") for m in medications if m.get("display")]

encounter = update_stage(encounter_id, "discharge", {
    "summary": discharge_summary,
    "icd10_codes": icd10_suggestions,
    "medications_at_discharge": meds_at_discharge,
    "follow_up": "Follow-up appointment in 1 week. Return sooner if symptoms worsen."
})

print(f"Encounter {encounter_id} updated with discharge summary.")
print(f"\nFull patient journey complete!")

## 6. View Complete Encounter Record

In [ ]:
final_encounter = load_encounter(encounter_id)

print("=" * 70)
print(f"COMPLETE ENCOUNTER RECORD: {encounter_id}")
print(f"Patient: {final_encounter['patient']['name']}")
print("=" * 70)

stages = final_encounter["stages"]

# Consultation
print("\n" + "-"*50)
print("STAGE 1: CONSULTATION")
print("-"*50)
print(f"Timestamp: {stages['consultation'].get('timestamp', 'N/A')}")
print(f"Transcript: {stages['consultation'].get('transcript', 'N/A')[:200]}...")
soap = stages['consultation'].get('soap_note', {})
if isinstance(soap, dict):
    for k, v in soap.items():
        if v:
            print(f"  {k.upper()}: {str(v)[:150]}...")

# Admission
print("\n" + "-"*50)
print("STAGE 2: ADMISSION")
print("-"*50)
print(f"Timestamp: {stages['admission'].get('timestamp', 'N/A')}")
print(f"Vitals: {len(stages['admission'].get('vitals_history', []))} readings")
print(f"Conditions: {len(stages['admission'].get('conditions', []))}")
print(f"Admission Note: {str(stages['admission'].get('admission_note', 'N/A'))[:200]}...")

# Radiology
print("\n" + "-"*50)
print("STAGE 3: RADIOLOGY")
print("-"*50)
print(f"Timestamp: {stages['radiology'].get('timestamp', 'N/A')}")
print(f"Images: {len(stages['radiology'].get('images', []))}")
for r in stages['radiology'].get('reports', []):
    print(f"  Report: {str(r.get('findings', 'N/A'))[:200]}...")

# Discharge
print("\n" + "-"*50)
print("STAGE 4: DISCHARGE")
print("-"*50)
print(f"Timestamp: {stages['discharge'].get('timestamp', 'N/A')}")
print(f"ICD-10 Codes: {stages['discharge'].get('icd10_codes', [])}")
print(f"Medications: {stages['discharge'].get('medications_at_discharge', [])}")
print(f"Follow-up: {stages['discharge'].get('follow_up', 'N/A')}")
print(f"\nDischarge Summary:\n{str(stages['discharge'].get('summary', 'N/A'))[:500]}...")

## 7. Export Final Document

In [ ]:
# Save the full encounter as a formatted markdown document
output_path = f"/content/encounters/{encounter_id}_discharge_report.md"

with open(output_path, "w") as f:
    f.write(f"# Discharge Report: {encounter_id}\n\n")
    f.write(f"**Patient:** {final_encounter['patient']['name']}\n")
    f.write(f"**MRN:** {final_encounter['patient']['mrn']}\n")
    f.write(f"**DOB:** {final_encounter['patient']['dob']}\n\n")
    f.write("---\n\n")
    f.write("## Consultation SOAP Note\n\n")
    if isinstance(soap, dict):
        for k, v in soap.items():
            if v:
                f.write(f"**{k.title()}:** {v}\n\n")
    f.write("---\n\n")
    f.write("## Admission Note\n\n")
    f.write(str(stages['admission'].get('admission_note', 'N/A')) + "\n\n")
    f.write("---\n\n")
    f.write("## Radiology Reports\n\n")
    for r in stages['radiology'].get('reports', []):
        f.write(str(r.get('findings', 'N/A')) + "\n\n")
    f.write("---\n\n")
    f.write("## Discharge Summary\n\n")
    f.write(str(stages['discharge'].get('summary', 'N/A')) + "\n\n")
    f.write("---\n\n")
    f.write("## ICD-10 Codes\n\n")
    for code in stages['discharge'].get('icd10_codes', []):
        f.write(f"- **{code['code']}**: {code['description']}\n")
    f.write("\n")

print(f"Discharge report exported to: {output_path}")
print("\nYou can download this file from the Colab file browser.")

# Also save the full JSON
json_path = f"/content/encounters/{encounter_id}.json"
print(f"Full encounter JSON: {json_path}")